In [1]:
import torch

In [2]:
# check cuda
torch.cuda.is_available()

False

In [3]:
# check mac supports pytorch acceleration
torch.backends.mps.is_available()

True

In [4]:
tensor0d = torch.tensor(1)                #1

print(tensor0d.shape)

tensor1d = torch.tensor([1, 2, 3])        #2

print(tensor1d.shape)

tensor2d = torch.tensor([[1, 2],
                         [3, 4]])         #3

print(tensor2d.shape)

tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])  #4

print(tensor3d.shape)

torch.Size([])
torch.Size([3])
torch.Size([2, 2])
torch.Size([2, 2, 2])


In [5]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

torch.int64


In [6]:
# for python floats pytorch creats tensors with 32-bit precision by default
# as 32-bit offers sufficient precision, while consuming less memory

floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


In [7]:
floatvec = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
print(floatvec.dtype)

torch.float64


In [8]:
floatvec = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
changed_type = floatvec.to(torch.float32)
print(changed_type.dtype)

torch.float32


In [9]:
tensor2d = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

In [10]:
tensor2d.shape

torch.Size([2, 3])

In [11]:
print(tensor2d.reshape(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [12]:
print(tensor2d.view(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [13]:
# transpose
print(tensor2d.T)

tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [14]:
print(tensor2d.matmul(tensor2d.T))

tensor([[14, 32],
        [32, 77]])


In [15]:
print(tensor2d @ tensor2d.T)

tensor([[14, 32],
        [32, 77]])


### Autograd System

In [16]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

print(f'{z=}')
print(f'{a=}')
print(f'{loss=}')

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

z=tensor([2.4200], grad_fn=<AddBackward0>)
a=tensor([0.9183], grad_fn=<SigmoidBackward0>)
loss=tensor(0.0852, grad_fn=<BinaryCrossEntropyBackward0>)


In [17]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [18]:
print(w1.grad)
print(b.grad)

None
None


In [19]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


Custom sigmoid and binary_cross_entropy functions

In [20]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def binary_cross_entropy(a, y):
    # a = prediction (sigmoid output), y = target (0 or 1)
    eps = 1e-12  # to avoid log(0)
    return -(y * math.log(a + eps) + (1 - y) * math.log(1 - a + eps))


In [21]:
y = 1.0
x1 = 1.1
w1 = 2.2
b = 0.0


z = x1 * w1 + b
a = sigmoid(z)
loss = binary_cross_entropy(a, y)

print(f'{z=}')
print(f'{a=}')
print(f'{loss=}')

z=2.4200000000000004
a=0.9183397445384054
loss=0.08518786473797667


micrograd reproduction of gradients

In [22]:
import sys
sys.path.append('../src')
from micrograd import Value

In [23]:
def sigmoid(z: Value):
    return 1 / (1 + (-z).exp())

def binary_cross_entropy(a, y):
    # a = prediction (sigmoid output), y = target (0 or 1)
    eps = 1e-12  # to avoid log(0)
    return -(y * (a + eps).log() + (1 - y) * (1 - a + eps).log())

In [24]:

y = Value(1.0)
x1 = Value(1.1)
w1 = Value(2.2)
b = Value(0.0)

z = x1 * w1 + b
a = sigmoid(z)

loss = binary_cross_entropy(a, y)

print(f'{z=}')
print(f'{a=}')
print(f'{loss=}')

z=Value(data=2.4200000000000004)
a=Value(data=0.9183397445384054)
loss=Value(data=0.08518786473797667)


In [25]:
loss.backward()
print(w1.grad)
print(b.grad)

-0.08982628100765629
-0.08166025546150571
